# 🏛️ POC 17: Half-Century Multi-Decade Walk-Forward Backtest (1978–2026)

**File**: [`research/notebooks/algo-alpha-execution/17_half_century_1978_2026_walkforward_backtest.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/17_half_century_1978_2026_walkforward_backtest.ipynb)  
**Scope**: Continuous step-by-step walk-forward backtest evaluating quantitative machine learning alpha across **nearly a half-century (48.6 Years: 1978–2026 / 12,250 daily trading sessions)**.

---

### Strategy Benchmark Matrix:
1. **`1. S&P 500 Index (^GSPC Benchmark)`**: Passive broad market benchmark.
2. **`2. Buy & Hold Equal-Weight (Static Top Universe)`**: Equal-weight static stock basket.
3. **`3. Naive XGBoost Default (30d Rebalance, 5d Forward Target, Equal Weight)`**: Baseline 5-day model with equal-weight allocation.
4. **`4. Naive XGBoost (30d Rebalance, 30d Forward Target, Forecast-Proportional Sizing)`**: Monthly fundamental drift model with conviction weighting.
5. **`5. Naive XGBoost (15d Rebalance, 15d Forward Target, Forecast-Proportional Sizing)`**: Bi-weekly swing momentum model with conviction weighting.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ HALF-CENTURY WALK-FORWARD TIMELINE (1978–2026 / 48.6 YEARS)                            │
│ 1978–1981: Initial Historical Burn-in & Feature Priming                                │
│ 1981–1989: 1980s Stagflation Recovery & 1987 Black Monday Crash                        │
│ 1990–1999: 1990s Tech Revolution & Dot-Com Mania                                      │
│ 2000–2009: Dot-Com Crash, 2008 Global Financial Crisis & Great Recession               │
│ 2010–2019: Post-Crisis Quantitative Easing Bull Run & Tech Dominance                  │
│ 2020–2026: COVID Shock, 2022 Inflation/Rate Hikes & GenAI Secular Wave                │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Environment Configuration

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "master_panel_1975_2026.parquet")
LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Loading Half-Century Master Parquet: {DATA_PATH}")

t0 = time.perf_counter()
df_master = pd.read_parquet(DATA_PATH)
df_master['date'] = pd.to_datetime(df_master['date'])

# Precompute forward prediction horizons
df_master['target_fwd_5d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-5) / s - 1.0)
df_master['target_fwd_15d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-15) / s - 1.0)
df_master['target_fwd_30d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-30) / s - 1.0)

print(f"✅ Loaded {len(df_master):,} records across {df_master['ticker'].nunique()} tickers ({df_master['date'].min().strftime('%Y-%m-%d')} to {df_master['date'].max().strftime('%Y-%m-%d')}) in {time.perf_counter()-t0:.2f}s!")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Loading Half-Century Master Parquet: c:\Users\honza\Desktop\projects\stock-analysis\data\processed\master_panel_1975_2026.parquet


✅ Loaded 638,434 records across 60 tickers (1978-01-03 to 2026-08-27) in 0.55s!


## 2. Ingest S&P 500 (`^GSPC`) Benchmark & Base Price Matrices (1978–2026)

In [2]:
prices_pivot = df_master.pivot(index='date', columns='ticker', values='close').ffill().bfill()
daily_rets = prices_pivot.pct_change().fillna(0.0)
all_dates = prices_pivot.index
start_dt = all_dates[0].strftime('%Y-%m-%d')
end_dt = all_dates[-1].strftime('%Y-%m-%d')

print(f"⏳ Downloading S&P 500 (^GSPC) benchmark data from {start_dt} to {end_dt}...")
spx_raw = yf.download("^GSPC", start=start_dt, end=end_dt, progress=False)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

spx_aligned = spx_raw['Close'].reindex(all_dates).ffill().bfill()
spx_equity = (spx_aligned / spx_aligned.iloc[0]) * 100.0

# Buy & Hold Equal Weight (Static Top Universe)
static_basket = daily_rets.mean(axis=1)
buy_hold_equity = (1.0 + static_basket).cumprod() * 100.0

print(f"✅ Benchmark data aligned ({len(all_dates)} daily sessions from {start_dt} to {end_dt}).")

⏳ Downloading S&P 500 (^GSPC) benchmark data from 1978-01-03 to 2026-08-27...


✅ Benchmark data aligned (12264 daily sessions from 1978-01-03 to 2026-08-27).


## 3. Half-Century Continuous Walk-Forward Simulation (1981–2026)

In [3]:
features = [
    'revenue_growth', 'net_margin', 'sentiment_score', 'rsi_14', 'macd',
    'is_opp_buy', 'is_pol_buy', 'ewma_volatility', 'daily_news_count',
    'daily_news_finbert_sentiment', 'news_volume_intensity',
    'news_decay_tau_1d_ema', 'news_decay_tau_3d_ema', 'news_sentiment_velocity'
]

# Set burn-in cutoff to 1981-01-02 (giving 3 years of initial training history 1978–1980)
burnin_end_date = pd.to_datetime('1981-01-02')
daily_rets_mat = daily_rets.values
n_days, n_tickers = daily_rets_mat.shape

w_naive_30_5 = np.zeros_like(daily_rets_mat)
w_naive_30_30 = np.zeros_like(daily_rets_mat)
w_naive_15_15 = np.zeros_like(daily_rets_mat)

# -----------------------------------------------------------------------------
# 1. 30-DAY REBALANCE WALK-FORWARD (1981–2026: ~370 Cycles)
# -----------------------------------------------------------------------------
F30 = 31
rebal_dates_30 = [d for d in all_dates[::F30] if d >= burnin_end_date]
print(f"🚀 Running Half-Century 30-Day Rebalance Walk-Forward ({len(rebal_dates_30)} cycles across 45.6 years)...")

t0_loop30 = time.perf_counter()

for reb_date in tqdm(rebal_dates_30, desc="30-Day Cycles"):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + F30, n_days)
    
    hist_df = df_master[(df_master['date'] < reb_date)]
    cand_df = df_master[df_master['date'] == reb_date]
    train_slice = hist_df.tail(120000)
    
    # 3. Naive XGBoost Default (30d Rebal / 5d Fwd / Equal Weight)
    train_5d = train_slice[train_slice['target_fwd_5d'].notnull()]
    m_5d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_5d.fit(train_5d[features], train_5d['target_fwd_5d'])
    p_5d = pd.Series(m_5d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df))).index
    w_eq = np.full(len(p_5d), 1.0 / len(p_5d))
    idx_5d = [prices_pivot.columns.get_loc(s) for s in p_5d if s in prices_pivot.columns]
    w_naive_30_5[t_idx:end_idx, idx_5d] = w_eq[:len(idx_5d)]
    
    # 4. Naive XGBoost (30d Rebal / 30d Fwd / Forecast-Proportional Sizing)
    train_30d = train_slice[train_slice['target_fwd_30d'].notnull()]
    m_30d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_30d.fit(train_30d[features], train_30d['target_fwd_30d'])
    p_30d = pd.Series(m_30d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_30d = p_30d.clip(lower=0.0001)
    w_prop_30d = (sc_30d / sc_30d.sum()).values
    idx_30d = [prices_pivot.columns.get_loc(s) for s in p_30d.index if s in prices_pivot.columns]
    w_naive_30_30[t_idx:end_idx, idx_30d] = w_prop_30d[:len(idx_30d)]

print(f"✅ 30-Day Cycles completed in {time.perf_counter() - t0_loop30:.2f}s!")

# -----------------------------------------------------------------------------
# 2. 15-DAY REBALANCE WALK-FORWARD (1981–2026: ~760 Cycles)
# -----------------------------------------------------------------------------
F15 = 15
rebal_dates_15 = [d for d in all_dates[::F15] if d >= burnin_end_date]
print(f"🚀 Running Half-Century 15-Day Rebalance Walk-Forward ({len(rebal_dates_15)} cycles across 45.6 years)...")

t0_loop15 = time.perf_counter()

for reb_date in tqdm(rebal_dates_15, desc="15-Day Cycles"):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + F15, n_days)
    
    hist_df = df_master[(df_master['date'] < reb_date)]
    cand_df = df_master[df_master['date'] == reb_date]
    train_slice = hist_df.tail(120000)
    
    # 5. Naive XGBoost (15d Rebal / 15d Fwd / Forecast-Proportional Sizing)
    train_15d = train_slice[train_slice['target_fwd_15d'].notnull()]
    m_15d = xgb.XGBRegressor(n_estimators=45, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist')
    m_15d.fit(train_15d[features], train_15d['target_fwd_15d'])
    p_15d = pd.Series(m_15d.predict(cand_df[features]), index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_15d = p_15d.clip(lower=0.0001)
    w_prop_15d = (sc_15d / sc_15d.sum()).values
    idx_15d = [prices_pivot.columns.get_loc(s) for s in p_15d.index if s in prices_pivot.columns]
    w_naive_15_15[t_idx:end_idx, idx_15d] = w_prop_15d[:len(idx_15d)]

print(f"✅ 15-Day Cycles completed in {time.perf_counter() - t0_loop15:.2f}s!")

🚀 Running Half-Century 30-Day Rebalance Walk-Forward (371 cycles across 45.6 years)...


30-Day Cycles:   0%|          | 0/371 [00:00<?, ?it/s]

✅ 30-Day Cycles completed in 188.91s!
🚀 Running Half-Century 15-Day Rebalance Walk-Forward (767 cycles across 45.6 years)...


15-Day Cycles:   0%|          | 0/767 [00:00<?, ?it/s]

✅ 15-Day Cycles completed in 215.41s!


## 4. Multi-Decade Performance Analytics & Risk Metrics (1981–2026)

In [4]:
eval_mask = (all_dates >= burnin_end_date)
eval_dates = all_dates[eval_mask]
eval_start_idx = all_dates.get_loc(burnin_end_date)

curves = {
    '1. S&P 500 Index (^GSPC Benchmark)': (spx_aligned.loc[eval_dates] / spx_aligned.loc[eval_dates].iloc[0]) * 100.0,
    '2. Buy & Hold Equal-Weight (Static Top Universe)': (buy_hold_equity.loc[eval_dates] / buy_hold_equity.loc[eval_dates].iloc[0]) * 100.0,
    '3. Naive XGBoost Default (30d Rebal, 5d Fwd, Equal Weight)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_30_5[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '4. Naive XGBoost (30d Rebal, 30d Fwd, Forecast Sizing)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_30_30[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '5. Naive XGBoost (15d Rebal, 15d Fwd, Forecast Sizing)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_naive_15_15[eval_start_idx:], axis=1)) * 100.0, index=eval_dates)
}

df_master_curves = pd.DataFrame(curves, index=eval_dates).reset_index().rename(columns={'index': 'date'})

def compute_analytics(series, spx_series, rf=0.03):
    r_strat = series.pct_change().dropna()
    r_spx = spx_series.pct_change().dropna()
    aligned = pd.concat([r_strat, r_spx], axis=1).dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    spx_cagr = (spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    alpha = (cagr - rf) - beta * (spx_cagr - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

analytics_records = []
for name, s in curves.items():
    analytics_records.append({'Strategy / Model': name, **compute_analytics(s, curves['1. S&P 500 Index (^GSPC Benchmark)'])})

df_performance_table = pd.DataFrame(analytics_records)
print("=== HALF-CENTURY (1981–2026) WALK-FORWARD PERFORMANCE & RISK MATRIX ===")
df_performance_table

=== HALF-CENTURY (1981–2026) WALK-FORWARD PERFORMANCE & RISK MATRIX ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,1. S&P 500 Index (^GSPC Benchmark),5.529823e+03,9.230016,0.416663,0.523635,-56.775388,0.162571,1.000000,0.000000
1,2. Buy & Hold Equal-Weight (Static Top Universe),1.162323e+05,16.721168,0.852409,1.080956,-44.980866,0.371740,0.875643,8.265901
2,"3. Naive XGBoost Default (30d Rebal, 5d Fwd, E...",2.821314e+05,19.009163,0.898911,1.160765,-45.983834,0.413388,0.964648,9.999393
3,"4. Naive XGBoost (30d Rebal, 30d Fwd, Forecast...",1.600017e+06,23.619141,1.019746,1.346418,-45.950521,0.514012,1.042278,14.125734
4,"5. Naive XGBoost (15d Rebal, 15d Fwd, Forecast...",1.541809e+06,23.518848,0.975421,1.258454,-62.511902,0.376230,1.069167,13.857920


## 5. Interactive Half-Century Visualizer (Log Scale: 1981–2026) & Drawdowns

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Half-Century Walk-Forward Equity Curves (Log Scale: 1981–2026 / 45.6 Years)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

palette = [
    '#636EFA',  # 1. SP500
    '#FFA15A',  # 2. Buy & Hold
    '#AB63FA',  # 3. Naive 30_5
    '#00CC96',  # 4. Naive 30_30 (Hero 1)
    '#FFDF00'   # 5. Naive 15_15 (Hero 2)
]

for idx, (name, s) in enumerate(curves.items()):
    c = palette[idx % len(palette)]
    is_hero = ('4.' in name or '5.' in name)
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=s, name=name,
        line=dict(color=c, width=3.0 if is_hero else 1.8)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_curves['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=c, width=1.5)
    ), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig.update_layout(
    template='plotly_dark', width=1200, height=850,
    title='<b>Half-Century Multi-Decade Benchmark (1981–2026): Passive vs. Naive XGBoost vs. Optimal Rebalance</b>',
    margin=dict(l=60, r=320, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig.show()

## 6. Decade-by-Decade Quantitative Breakdown

In [6]:
decades = [
    ('1980s (1981–1989)', pd.to_datetime('1981-01-02'), pd.to_datetime('1989-12-29')),
    ('1990s (1990–1999)', pd.to_datetime('1990-01-02'), pd.to_datetime('1999-12-31')),
    ('2000s (2000–2009)', pd.to_datetime('2000-01-03'), pd.to_datetime('2009-12-31')),
    ('2010s (2010–2019)', pd.to_datetime('2010-01-04'), pd.to_datetime('2019-12-31')),
    ('2020s (2020–2026)', pd.to_datetime('2020-01-02'), pd.to_datetime('2026-08-27'))
]

decade_records = []
for dec_name, d_start, d_end in decades:
    row = {'Decade': dec_name}
    for name, s in curves.items():
        s_sub = s[(s.index >= d_start) & (s.index <= d_end)]
        if len(s_sub) > 0:
            sub_ret = (s_sub.iloc[-1] / s_sub.iloc[0] - 1.0) * 100.0
            row[name.split('.')[1].strip().split('(')[0].strip()] = round(sub_ret, 2)
    decade_records.append(row)

df_decades = pd.DataFrame(decade_records)
print("=== DECADE-BY-DECADE TOTAL RETURN BREAKDOWN (%) ===")
df_decades

=== DECADE-BY-DECADE TOTAL RETURN BREAKDOWN (%) ===


,Decade,S&P 500 Index,Buy & Hold Equal-Weight,Naive XGBoost Default,Naive XGBoost
0,1980s (1981–1989),159.20,293.83,561.29,1066.56
1,1990s (1990–1999),308.48,854.09,1330.07,2244.93
2,2000s (2000–2009),-23.37,144.12,148.77,199.26
3,2010s (2010–2019),185.16,339.26,288.03,329.30
4,2020s (2020–2026),135.61,180.49,198.44,317.11


## 7. Export Half-Century Walk-Forward Simulation to Excel

In [7]:
out_path = os.path.join(LOCAL_DATA_DIR, "half_century_1981_2026_walkforward_simulation_poc.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_master_curves.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_performance_table.to_excel(writer, sheet_name='performance_summary', index=False)
    df_decades.to_excel(writer, sheet_name='decade_breakdown', index=False)

print(f"💾 Successfully exported Half-Century Walk-Forward Benchmark to: {out_path}")

💾 Successfully exported Half-Century Walk-Forward Benchmark to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\half_century_1981_2026_walkforward_simulation_poc.xlsx
